In [1]:
import os
import cv2
import glob
import json
import pickle
import random
import numpy as np
from tqdm import tqdm
import random
random.seed(20)

In [2]:
LABELS_M2 = [
    "R1", "R2", "R3", "R4", # crops
    "RM1", "RM2",
    "RSL1", "RSM1",
    "LTE", "MTE", # edges
    "LTL", "MTM",
    "LTC", "MTC", # center
    "LTM", "MTL"
]

LABELS_M = [
    "RSLT1", "RSMT1"
]

def get_keypoints(filepath, LABELS):
    keypoints = {}

    with open(filepath, "r") as f:
        text = f.read()
    f.close()

    for line in text.split("\n"):
        if any(line.startswith(l) for l in LABELS):
            items = line.split()
            keypoints[items[0]] = (float(items[1]), float(items[3]))
    
    return keypoints

In [3]:
implant_list = [
    "dataset/2021_07월/0701 유옥례_KP(RT불가)",
    "dataset/2021_05월/0525 최정필_KP(RT불가)",
    "dataset/2021_08월/0730 이경희_KP(RT불가)",
    "dataset/2018_11월/0824 이정이_KP(RT불가)",
    "dataset/2021_05월/0510 박금순_KP(RT불가)",
    "dataset/2019_11월/1120 문희순_KP(RT불가)",
    "dataset/2021_07월/0623 조정현_KP(RT불가)",
    "dataset/2021_02월/0215 최효심_KP(RT불가)",
    "dataset/2018_3월/0227 조만순",
    "dataset/2020_05월/0527 김명국_KP_RT",
    "dataset/2021_08월/0812 이순임_KP",
    "dataset/2021_07월/0707 엄연옥_KP(RT불가)",
    "dataset/2020_01월/1228 손윤자_KP_LT(RT불가)",
    "dataset/2021_07월/0708 하금녀_KP(RT불가)",
    "dataset/2018_9월/0903 조원당",
    "dataset/2020_11월/1109 정원대_KP_LT",
    "dataset/2020_01월/0117 명태원_KP_LT(RT불가)",
    "dataset/2020_07월/0723 김경옥_KP_LT(RT불가)",
    "dataset/2021_10월/1006 여경자_KP(RT불가)",
    "dataset/2021_06월/0608 김영월_KP(RT불가)",
    "dataset/2019_09월/0905 문준회_KP(RT불가)",
    "dataset/2019_10월/0921 조명심_KP(RT불가)",
    "dataset/2021_10월/1004 이영숙_KP(RT불가)",
    "dataset/2020_09월/0904 박병심_KP_LT(RT불가)",
    "dataset/2021_02월/0118 신건수_KP(RT불가)",
    "dataset/2020_02월/0129 김순자_KP_LT(RT불가)",
    "dataset/2018_4월/0404 유순자(RT불가)",
    "dataset/2018_4월/0409 김말순(RT불가)",
    "dataset/2018_4월/0410 김범국(RT불가)",
    "dataset/2020_12월/1119 오점숙_KP(불가)",
    "dataset/2021_02월/1109 박정순_KP(RT불가)",
    "dataset/2021_01월/1128 박애선_KP(RT불가)",
    "dataset/2020_09월/0915 유문태_KP_LT(RT불가)",
    "dataset/2019_11월/1026 서미경_KP(RT불가)",
    "dataset/2019_10월/0706 윤인자_KP(RT불가)",
    "dataset/2020_11월/1107 정정숙_KP_LT(RT불가)",
    "dataset/2020_09월/0928 윤순덕_KP_LT(RT불가)",
    "dataset/2021_01월/0128 김영자_KP(RT불가)",
    "dataset/2020_01월/1219 박옥순_KP_LT(RT불가)",
    "dataset/2019_09월/0909 편종범_KP(RT불가)",
    "dataset/2021_06월/1124 김기선_KP(RT불가)",
    "dataset/2019_09월/0910 강인순_KP(RT불가)",
    "dataset/2019_09월/0617 김동자_KP",
    "dataset/2019_11월/1102 이영희_KP(RT불가)",
    "dataset/2021_06월/0610 김진순_KP(RT불가)",
    "dataset/2019_10월/0928 박행규_KP(RT불가)",
    "dataset/2020_08월/0812 문순자_KP_LT(RT불가)"
]
ankle_list = [
    "dataset/2019_10월/1002 한순희_KP(RT불가)",
    "dataset/2020_01월/1221 박순자_KP_LT(x-ray잘림)_불가",
    "dataset/2018_11월/1114 조종순(불가)"
]

exclude_list = [ 
    "dataset/2019_10월/0803 손정이_KP",
    "dataset/2021_05월/0526 이기순_KP",
    "dataset/2018_8월/0210 박임성",
    "dataset/2021_02월/0202 김정순_KP",
    "dataset/2020_12월/1127 조광희_KP_RT",
    "dataset/2019_09월/0521 여운자_KP",
    "dataset/2019_10월/0808 임귀순_KP",
    "dataset/2019_11월/1108 유재욱_KP",
    "dataset/2020_01월/1221 박옥순_KP",
    "dataset/2020_12월/1218 방명욱_KP(x-ray잘림)_불가",
    "dataset/2020_12월/1109 박정순_KP_LT(RT불가)",
    "dataset/2018_11월/1102 한은혜",
    "dataset/2020_02월/0206 조경형_KP_LT(RT불가)",
    "dataset/2020_06월/0528 노정숙_KP"
]

In [4]:
skip_list = ankle_list + implant_list + exclude_list
print(len(skip_list))

64


In [5]:
folderpaths = glob.glob("dataset/*/*")

data = []

for path in folderpaths:
    if path in skip_list:
        continue
    jpg_files_m = glob.glob(os.path.join(path, "M" , "*.jpg"))
    txt_files_m = glob.glob(os.path.join(path, "M" , "*.txt"))

    jpg_files_m2 = glob.glob(os.path.join(path, "M2" , "*.jpg"))
    txt_files_m2 = glob.glob(os.path.join(path, "M2" , "*.txt"))

    if len(jpg_files_m) > 1 or len(jpg_files_m2) > 1:
        print(f"Found multiple jpg files -> {path}")
    if len(txt_files_m) > 1 or len(txt_files_m2) > 1:
        print(f"Found multiple txt files: {path}")
    
    try:
        img_path = jpg_files_m2[0]
        keypoints_m = get_keypoints(txt_files_m[0], LABELS_M)
        keypoints_m2 = get_keypoints(txt_files_m2[0], LABELS_M2)

        for l in LABELS_M:
            if l not in keypoints_m:
                print(f"{l} is missing: {path}")
        for l in LABELS_M2:
            if l not in keypoints_m2:
                print(f"{l} is missing: {path}")
        
        keypoints_m2.update(keypoints_m)
        data.append((img_path, keypoints_m2))
    except:
        print(f"Something wrong: {path}")

print(f"data size: {len(data)}")

data size: 3104


In [6]:
random.shuffle(data)
data = data[:2000]

In [8]:
total = len(data)
train_end = int(total * 0.8)
val_end = int(total * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

In [10]:
len(val_data)

200

In [11]:
with open("pickle/train.pkl", "wb") as f:
    pickle.dump(train_data, f)

with open("pickle/val.pkl", "wb") as f:
    pickle.dump(val_data, f)

with open("pickle/test.pkl", "wb") as f:
    pickle.dump(test_data, f)

In [ ]:
# with open("pickle/folds.pkl", "rb") as f:
#     folds = pickle.load(f)
# val_data = sum([fold for i, fold in enumerate(folds)], [])
# print(len(val_data))

In [6]:
fold_size = len(data) // 5
print(f"fold_size: {fold_size}")
folds = [data[i*fold_size:(i+1)*fold_size] for i in range(5)]

with open("pickle/folds.pkl", "wb") as f:
    pickle.dump(folds, f)

fold_size: 626
